In [13]:
# Impoting packages
import numpy as np
import pandas as pd
import os
import kagglehub
import os
import json
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")

print("Path to dataset files:", path)
archivos = os.listdir(path)

print("Los archivos descargados son:")
for archivo in archivos:
    print("-", archivo)

Path to dataset files: /home/josema/.cache/kagglehub/datasets/fronkongames/steam-games-dataset/versions/31
Los archivos descargados son:
- games.csv
- games.json


In [14]:
# 1. Leer el archivo JSON original
dataset = {}
if os.path.exists(path+'/games.json'):
    with open(path+'/games.json', 'r', encoding='utf-8') as fin:
        text = fin.read()
        if len(text) > 0:
            dataset = json.loads(text)

# 2. Lista para almacenar cada fila del dataset
rows = []

for appID, game in dataset.items():
    # Creamos un diccionario plano para cada juego
    row = {
    'Appid': appID,
    'Name': game.get('name'),
    'Release_date': game.get('release_date'),
    'Estimated_owners': game.get('estimated_owners'),
    'Peak_ccu': game.get('peak_ccu'),
    'Required_age': game.get('required_age'),
    'Price': game.get('price'),
    'Dlc_count': game.get('dlc_count'),
    'Short_description': game.get('short_description'),
    'Languages': game.get('supported_languages'),
    'Full_audio_languages': game.get('full_audio_languages'),
    'Reviews': game.get('reviews'),
    'Windows': game.get('windows'),
    'Mac': game.get('mac'),
    'Linux': game.get('linux'),
    'Metacritic_score': game.get('metacritic_score'),
    'Metacritic_url': game.get('metacritic_url'),
    'User_score': game.get('user_score'),
    'Positive': game.get('positive'),
    'Negative': game.get('negative'),
    'Score_rank': game.get('score_rank'),
    'Achievements': game.get('achievements'),
    'Recommendations': game.get('recommendations'),
    'Average_playtime_forever': game.get('average_playtime_forever'),
    'Average_playtime_2weeks': game.get('average_playtime_2weeks'),
    'Median_playtime_forever': game.get('median_playtime_forever'),
    'Median_playtime_2weeks': game.get('median_playtime_2weeks'),

    'Developers': ", ".join(game.get('developers', [])) if game.get('developers') else "",
    'Publishers': ", ".join(game.get('publishers', [])) if game.get('publishers') else "",
    'Categories': ", ".join(game.get('categories', [])) if game.get('categories') else "",
    'Genres': ", ".join(game.get('genres', [])) if game.get('genres') else "",
    'Tags': ", ".join(game.get('tags', {}).keys()) if isinstance(game.get('tags'), dict) else ""
    }

    
    # Añadimos la fila procesada a la lista
    rows.append(row)

# 3. Convertir la lista de filas en un DataFrame de Pandas
df = pd.DataFrame(rows)

In [15]:
def load_and_clean_data(df):
    #df = pd.read_parquet(PARQUET_PATH)
    
    # 1. Definir lista negra aquí mismo para tenerla a mano
    LISTA_NEGRA = [
        "Web Publishing", "Animation & Modeling", "Education", "Software Training", 
        "Accounting", "Video Production", "Utilities", "Audio Production", 
        "Design & Illustration", "Photo Editing", "Game Development", "Software"
    ]

    df['Score_rank'] = pd.to_numeric(df['Score_rank'], errors='coerce')

    # Optional: If you want to handle the NaNs (fill with 0 or drop them)
    df['Score_rank'] = df['Score_rank'].fillna(0).astype(int)
    
    df["Release_date"] = pd.to_datetime(df["Release_date"], errors="coerce")
    df["Year"] = df["Release_date"].dt.year
    df["Positive"] = pd.to_numeric(df["Positive"], errors="coerce").fillna(0)
    df["Negative"] = pd.to_numeric(df["Negative"], errors="coerce").fillna(0)
    df["Total_Reviews"] = df["Positive"] + df["Negative"]
    df["Aceptacion_Real"] = (df["Positive"] / df["Total_Reviews"] * 100).fillna(0)
    
    df["Median_playtime_forever"] = pd.to_numeric(df["Median_playtime_forever"], errors="coerce").fillna(0)
    df["Median_playtime_2weeks"] = pd.to_numeric(df["Median_playtime_2weeks"], errors="coerce").fillna(0)
    
    # 2. Limpieza de Géneros (Procesamos primero)
# 2. Función de limpieza que filtra la lista al vuelo
    def procesar_generos_limpios(x):
        if pd.isna(x): return []
        # Limpia espacios Y elimina los que están en la lista negra
        return [g.strip() for g in str(x).split(",") if g.strip() and g.strip() not in LISTA_NEGRA]
    
    df["Genres_list"] = df["Genres"].apply(procesar_generos_limpios)
    
    # 3. Ahora el Género Principal ya no puede ser software
    df["Genero_Principal"] = df["Genres_list"].apply(lambda x: x[0] if x else "Otros")
    
    # 4. Filtro de filas (opcional, pero recomendado)
    df = df[df["Genres_list"].apply(len) > 0]
    
    # 3. FILTRADO GLOBAL: Eliminamos software y basura de un solo golpe
    df = df[~df["Genero_Principal"].isin(LISTA_NEGRA)]
    df = df[(df["Total_Reviews"] > 0) | (df["Median_playtime_forever"] > 0)]
    
    # Limpiar Estimated_owners (convertir rango a valor numérico conservador)
    def extraer_min_owners(x):
        if pd.isna(x) or str(x) == '0 - 0': return 0
        try: return float(str(x).split('-')[0].strip())
        except: return 0

        # Asegurar que el precio es float limpio
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
    # Crear una bandera para diferenciar F2P fácilmente en tus gráficas
    df["Is_Free"] = df["Price"] == 0


    # Limpieza básica para evitar duplicados en filtros
    df["Developers"] = df["Developers"].fillna("Unknown").str.strip()
    df["Publishers"] = df["Publishers"].fillna("Unknown").str.strip()

    # Si hay duplicados, nos quedamos con la última entrada (o la que tenga más reviews)
    df = df.sort_values("Total_Reviews", ascending=False).drop_duplicates(subset="Appid", keep="first")

    df["Estimated_owners_min"] = df["Estimated_owners"].apply(extraer_min_owners)
    
    # Cálculos de negocio (Hecho una vez aquí)
    df["Mercado_Estimado"] = df["Estimated_owners_min"] * df["Price"]
    
    df["Tag_List"] = df["Tags"].apply(lambda x: [t.strip() for t in str(x).split(",")] if pd.notna(x) else [])
    
    return df

df = load_and_clean_data(df)

# 4. Guardar el DataFrame a un archivo CSV
# Usamos index=False para que no guarde una columna extra con los números de fila
df.to_csv('games.csv', index=False, encoding='utf-8')

df.to_parquet("games.parquet")

print(f"¡Proceso completado! Se han procesado {len(df)} juegos y se ha guardado 'games.csv' & 'games.parquet'")


¡Proceso completado! Se han procesado 83912 juegos y se ha guardado 'games.csv' & 'games.parquet'


In [16]:
#Little data exploration
df = pd.read_parquet("games.parquet")


pd.set_option('display.max_columns', None)
df

,Appid,Name,Release_date,Estimated_owners,Peak_ccu,Required_age,Price,Dlc_count,Short_description,Languages,Full_audio_languages,Reviews,Windows,Mac,Linux,Metacritic_score,Metacritic_url,User_score,Positive,Negative,Score_rank,Achievements,Recommendations,Average_playtime_forever,Average_playtime_2weeks,Median_playtime_forever,Median_playtime_2weeks,Developers,Publishers,Categories,Genres,Tags,Year,Total_Reviews,Aceptacion_Real,Genres_list,Genero_Principal,Is_Free,Estimated_owners_min,Mercado_Estimado,Tag_List
45509,730,Counter-Strike 2,2012-08-21,100000000 - 200000000,1013936,0,0.00,1,"For over two decades, Counter-Strike has offer...","[Czech, Danish, Dutch, English, Finnish, Frenc...","[English, Indonesian]",,True,False,True,0,,0,7642084,1173003,0,1,4830455,33906,702,6237,307,Valve,Valve,"Multi-player, Cross-Platform Multiplayer, Stea...","Action, Free To Play","FPS, Shooter, Multiplayer, Competitive, Action...",2012,8815087,86.693234,"[Action, Free To Play]",Action,True,100000000.0,0.0,"[FPS, Shooter, Multiplayer, Competitive, Actio..."
8878,578080,PUBG: BATTLEGROUNDS,2017-12-21,100000000 - 200000000,314682,13,0.00,0,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...","[English, Korean, Simplified Chinese, French, ...",[],,True,False,False,0,,0,1520457,1037487,0,37,1753261,23787,730,6082,302,PUBG Corporation,"KRAFTON, Inc.","Multi-player, PvP, Online PvP, Stats, Remote P...","Action, Adventure, Massively Multiplayer, Free...","Survival, Shooter, Battle Royale, Multiplayer,...",2017,2557944,59.440590,"[Action, Adventure, Massively Multiplayer, Fre...",Action,True,100000000.0,0.0,"[Survival, Shooter, Battle Royale, Multiplayer..."
4193,570,Dota 2,2013-07-09,100000000 - 200000000,623941,0,0.00,2,"Every day, millions of players worldwide enter...","[Bulgarian, Czech, Danish, Dutch, English, Fin...","[English, Korean, Simplified Chinese, Vietnamese]",“A modern multiplayer masterpiece.” 9.5/10 – D...,True,True,True,90,https://www.metacritic.com/game/pc/dota-2?ftag...,0,2037143,461826,0,0,14369,54064,1427,1155,860,Valve,Valve,"Multi-player, Co-op, Steam Trading Cards, Stea...","Action, Strategy, Free To Play","Free to Play, MOBA, Multiplayer, Strategy, e-s...",2013,2498969,81.519339,"[Action, Strategy, Free To Play]",Action,True,100000000.0,0.0,"[Free to Play, MOBA, Multiplayer, Strategy, e-..."
59122,271590,Grand Theft Auto V Legacy,2015-04-13,50000000 - 100000000,67851,17,0.00,0,Grand Theft Auto V for PC offers players the o...,"[English, French, Italian, German, Spanish - S...","[English, Spanish - Latin America]",,True,False,False,96,https://www.metacritic.com/game/pc/grand-theft...,0,1739980,250576,0,77,1854262,14657,872,5846,158,Rockstar North,Rockstar Games,"Single-player, Multi-player, PvP, Online PvP, ...","Action, Adventure","Open World, Action, Multiplayer, Crime, Automo...",2015,1990556,87.411758,"[Action, Adventure]",Action,True,50000000.0,0.0,"[Open World, Action, Multiplayer, Crime, Autom..."
10927,105600,Terraria,2011-05-16,20000000 - 50000000,24580,0,4.99,2,"Dig, fight, explore, build! Nothing is impossi...","[English, French, Italian, German, Spanish - S...",[],,True,True,True,83,https://www.metacritic.com/game/pc/terraria?ft...,0,1373979,35494,0,115,1163012,8245,622,2051,180,Re-Logic,Re-Logic,"Single-player, Multi-player, PvP, Online PvP, ...","Action, Adventure, Indie, RPG","Open World Survival Craft, Sandbox, Survival, ...",2011,1409473,97.481754,"[Action, Adventure, Indie, RPG]",Action,False,20000000.0,99800000.0,"[Open World Survival Craft, Sandbox, Survival,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15997,2152540,Chronicles of Forgotten Tears,2023-03-10,0 - 20000,0,0,2.99,0,"Chronicles of Forgotten Tears, narrates the ev...","[English, Italian]",[],,True,False,False,0,,0,0,0,0,31,0,123,0,163,0,Old Owl Game,Old Owl Game,"Single-player, Steam Achievements, Family Sharing","Action,

In [17]:
df.Windows.value_counts()

Windows
True     83900
False       12
Name: count, dtype: int64

In [18]:
df.iloc[0]


Appid                                                                     730
Name                                                         Counter-Strike 2
Release_date                                              2012-08-21 00:00:00
Estimated_owners                                        100000000 - 200000000
Peak_ccu                                                              1013936
Required_age                                                                0
Price                                                                     0.0
Dlc_count                                                                   1
Short_description           For over two decades, Counter-Strike has offer...
Languages                   [Czech, Danish, Dutch, English, Finnish, Frenc...
Full_audio_languages                                    [English, Indonesian]
Reviews                                                                      
Windows                                                         

In [19]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Release_date,83912,2021-01-26 00:56:14.163409,1997-06-30 00:00:00,2018-10-06 18:00:00,2021-07-29 00:00:00,2023-11-09 00:00:00,2025-12-29 00:00:00,NaN
Peak_ccu,83912.0,78.586114,0.0,0.0,0.0,0.0,1013936.0,4506.244396
Required_age,83912.0,0.223353,0.0,0.0,0.0,0.0,21.0,1.902336
Price,83912.0,5.17158,0.0,0.99,2.99,5.99,999.98,11.322416
Dlc_count,83912.0,0.697481,0.0,0.0,0.0,0.0,3703.0,17.482625
Metacritic_score,83912.0,3.707944,0.0,0.0,0.0,0.0,97.0,16.294464
User_score,83912.0,0.035871,0.0,0.0,0.0,0.0,100.0,1.686032
Positive,83912.0,1519.969659,0.0,4.0,16.0,88.0,7642084.0,33942.285457
Negative,83912.0,246.351118,0.0,1.0,4.0,24.0,1173003.0,6495.275223
Score_rank,83912.0,0.047276,0.0,0.0,0.0,0.0,100.0,2.164856


In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 83912 entries, 45509 to 25487
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Appid                     83912 non-null  str           
 1   Name                      83912 non-null  str           
 2   Release_date              83912 non-null  datetime64[us]
 3   Estimated_owners          83912 non-null  str           
 4   Peak_ccu                  83912 non-null  int64         
 5   Required_age              83912 non-null  int64         
 6   Price                     83912 non-null  float64       
 7   Dlc_count                 83912 non-null  int64         
 8   Short_description         83912 non-null  str           
 9   Languages                 83912 non-null  object        
 10  Full_audio_languages      83912 non-null  object        
 11  Reviews                   83912 non-null  str           
 12  Windows                   8391

In [21]:
df.Tags.value_counts()


Tags
                                                                                                                                                                                                                                                                                                   2019
Adventure, Casual, Hidden Object                                                                                                                                                                                                                                                                    273
Action, Indie                                                                                                                                                                                                                                                                                       249
Indie, Casual                                                                                              

In [22]:
df.columns

Index(['Appid', 'Name', 'Release_date', 'Estimated_owners', 'Peak_ccu',
       'Required_age', 'Price', 'Dlc_count', 'Short_description', 'Languages',
       'Full_audio_languages', 'Reviews', 'Windows', 'Mac', 'Linux',
       'Metacritic_score', 'Metacritic_url', 'User_score', 'Positive',
       'Negative', 'Score_rank', 'Achievements', 'Recommendations',
       'Average_playtime_forever', 'Average_playtime_2weeks',
       'Median_playtime_forever', 'Median_playtime_2weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'Year', 'Total_Reviews',
       'Aceptacion_Real', 'Genres_list', 'Genero_Principal', 'Is_Free',
       'Estimated_owners_min', 'Mercado_Estimado', 'Tag_List'],
      dtype='str')

In [23]:
# Convertir Tags en listas limpias
df['Tag_List'] = df['Tags'].fillna('').apply(
    lambda x: [t.strip() for t in x.split(',') if t.strip()]
)

# Explode para separar cada tag en una fila
df_tags = df.explode('Tag_List')

# Lista única de tags
tags_unicos = sorted(df_tags['Tag_List'].dropna().unique())

print(len(tags_unicos))
print(tags_unicos)


452
['1980s', "1990's", '2.5D', '2D', '2D Fighter', '2D Platformer', '360 Video', '3D', '3D Fighter', '3D Platformer', '3D Vision', '4 Player Local', '4X', '6DOF', '8-bit Music', 'ATV', 'Abstract', 'Action', 'Action RPG', 'Action RTS', 'Action Roguelike', 'Action-Adventure', 'Addictive', 'Adventure', 'Agriculture', 'Aliens', 'Alternate History', 'Ambient', 'America', 'Animation & Modeling', 'Anime', 'Arcade', 'Archery', 'Arena Shooter', 'Artificial Intelligence', 'Assassin', 'Asymmetric VR', 'Asynchronous Multiplayer', 'Atmospheric', 'Audio Production', 'Auto Battler', 'Automation', 'Automobile Sim', 'BMX', 'Base-Building', 'Baseball', 'Based On A Novel', 'Basketball', 'Batman', 'Battle Royale', "Beat 'em up", 'Beautiful', 'Benchmark', 'Bikes', 'Birds', 'Blood', 'Board Game', 'Boomer Shooter', 'Boss Rush', 'Bowling', 'Boxing', 'Building', 'Bullet Hell', 'Bullet Time', 'CRPG', 'Capitalism', 'Card Battler', 'Card Game', 'Cartoon', 'Cartoony', 'Casual', 'Cats', 'Character Action Game', 'C

In [24]:
# Convertir Genres en listas limpias
df['Genre_List'] = df['Genres'].fillna('').apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip()]
)

# Explode para separar cada género en una fila
df_genres = df.explode('Genre_List')

# Lista única de géneros
genres_unicos = sorted(df_genres['Genre_List'].dropna().unique())

print(len(genres_unicos))
print(genres_unicos)


33
['360 Video', 'Accounting', 'Action', 'Adventure', 'Animation & Modeling', 'Audio Production', 'Casual', 'Design & Illustration', 'Documentary', 'Early Access', 'Education', 'Episodic', 'Free To Play', 'Game Development', 'Gore', 'Indie', 'Massively Multiplayer', 'Movie', 'Nudity', 'Photo Editing', 'RPG', 'Racing', 'Sexual Content', 'Short', 'Simulation', 'Software Training', 'Sports', 'Strategy', 'Tutorial', 'Utilities', 'Video Production', 'Violent', 'Web Publishing']


In [25]:
df['Tag_List']

45509     [FPS, Shooter, Multiplayer, Competitive, Actio...
8878      [Survival, Shooter, Battle Royale, Multiplayer...
4193      [Free to Play, MOBA, Multiplayer, Strategy, e-...
59122     [Open World, Action, Multiplayer, Crime, Autom...
10927     [Open World Survival Craft, Sandbox, Survival,...
                                ...                        
15997                                                    []
88967                                                    []
88963                                                    []
100913                                                   []
25487                                                    []
Name: Tag_List, Length: 83912, dtype: object

In [26]:
df['Release_date'] = pd.to_datetime(df['Release_date'], errors='coerce')
df['Release_Day'] = df['Release_date'].dt.day
df['Release_Month'] = df['Release_date'].dt.month
df['Release_Year'] = df['Release_date'].dt.year
print(df[['Name', 'Release_date', 'Release_Day', 'Release_Month', 'Release_Year']].head())

                            Name Release_date  Release_Day  Release_Month  \
45509           Counter-Strike 2   2012-08-21           21              8   
8878         PUBG: BATTLEGROUNDS   2017-12-21           21             12   
4193                      Dota 2   2013-07-09            9              7   
59122  Grand Theft Auto V Legacy   2015-04-13           13              4   
10927                   Terraria   2011-05-16           16              5   

       Release_Year  
45509          2012  
8878           2017  
4193           2013  
59122          2015  
10927          2011  


In [27]:
df.info()

<class 'pandas.DataFrame'>
Index: 83912 entries, 45509 to 25487
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Appid                     83912 non-null  str           
 1   Name                      83912 non-null  str           
 2   Release_date              83912 non-null  datetime64[us]
 3   Estimated_owners          83912 non-null  str           
 4   Peak_ccu                  83912 non-null  int64         
 5   Required_age              83912 non-null  int64         
 6   Price                     83912 non-null  float64       
 7   Dlc_count                 83912 non-null  int64         
 8   Short_description         83912 non-null  str           
 9   Languages                 83912 non-null  object        
 10  Full_audio_languages      83912 non-null  object        
 11  Reviews                   83912 non-null  str           
 12  Windows                   8391

In [28]:
df.columns


Index(['Appid', 'Name', 'Release_date', 'Estimated_owners', 'Peak_ccu',
       'Required_age', 'Price', 'Dlc_count', 'Short_description', 'Languages',
       'Full_audio_languages', 'Reviews', 'Windows', 'Mac', 'Linux',
       'Metacritic_score', 'Metacritic_url', 'User_score', 'Positive',
       'Negative', 'Score_rank', 'Achievements', 'Recommendations',
       'Average_playtime_forever', 'Average_playtime_2weeks',
       'Median_playtime_forever', 'Median_playtime_2weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'Year', 'Total_Reviews',
       'Aceptacion_Real', 'Genres_list', 'Genero_Principal', 'Is_Free',
       'Estimated_owners_min', 'Mercado_Estimado', 'Tag_List', 'Genre_List',
       'Release_Day', 'Release_Month', 'Release_Year'],
      dtype='str')

In [29]:
df[df['Name'] == 'CosmoDreamer']



,Appid,Name,Release_date,Estimated_owners,Peak_ccu,Required_age,Price,Dlc_count,Short_description,Languages,Full_audio_languages,Reviews,Windows,Mac,Linux,Metacritic_score,Metacritic_url,User_score,Positive,Negative,Score_rank,Achievements,Recommendations,Average_playtime_forever,Average_playtime_2weeks,Median_playtime_forever,Median_playtime_2weeks,Developers,Publishers,Categories,Genres,Tags,Year,Total_Reviews,Aceptacion_Real,Genres_list,Genero_Principal,Is_Free,Estimated_owners_min,Mercado_Estimado,Tag_List,Genre_List,Release_Day,Release_Month,Release_Year
88322,1424630,CosmoDreamer,2020-10-02,0 - 20000,0,0,4.79,0,From the popular free game 'Outside' comes a d...,"[English, Japanese, Simplified Chinese, Tradit...",[],,True,False,False,0,,0,121,0,0,40,128,0,0,0,0,あうとさいど,あうとさいど,"Single-player, Steam Achievements, Partial Con...",Action,"Bullet Hell, Action, Top-Down Shooter, 2D Plat...",2020,121,100.0,[Action],Action,False,0.0,0.0,"[Bullet Hell, Action, Top-Down Shooter, 2D Pla...",[Action],2,10,2020


In [30]:
df.Estimated_owners.unique()

<ArrowStringArray>
['100000000 - 200000000',  '50000000 - 100000000',   '20000000 - 50000000',
   '10000000 - 20000000',    '5000000 - 10000000',             '0 - 20000',
     '2000000 - 5000000',     '1000000 - 2000000',      '500000 - 1000000',
       '200000 - 500000',        '50000 - 100000',       '100000 - 200000',
         '20000 - 50000']
Length: 13, dtype: str

In [31]:
df[df['Name'].str.contains('DARK SOULS')]

,Appid,Name,Release_date,Estimated_owners,Peak_ccu,Required_age,Price,Dlc_count,Short_description,Languages,Full_audio_languages,Reviews,Windows,Mac,Linux,Metacritic_score,Metacritic_url,User_score,Positive,Negative,Score_rank,Achievements,Recommendations,Average_playtime_forever,Average_playtime_2weeks,Median_playtime_forever,Median_playtime_2weeks,Developers,Publishers,Categories,Genres,Tags,Year,Total_Reviews,Aceptacion_Real,Genres_list,Genero_Principal,Is_Free,Estimated_owners_min,Mercado_Estimado,Tag_List,Genre_List,Release_Day,Release_Month,Release_Year
111007,374320,DARK SOULS™ III,2016-04-11,5000000 - 10000000,4381,17,59.99,3,Dark Souls continues to push the boundaries wi...,"[English, French, Italian, German, Spanish - S...","[English, Korean]",“Dark Souls 3's incredible world and awe-inspi...,True,False,False,89,https://www.metacritic.com/game/pc/dark-souls-...,0,390326,23449,0,43,269868,5102,328,2365,155,"FromSoftware, Inc.","FromSoftware, Inc., Bandai Namco Entertainment","Single-player, Multi-player, Co-op, Steam Achi...",Action,"Souls-like, Dark Fantasy, Difficult, RPG, Atmo...",2016,413775,94.332910,[Action],Action,False,5000000.0,299950000.0,"[Souls-like, Dark Fantasy, Difficult, RPG, Atm...",[Action],11,4,2016
81392,570940,DARK SOULS™: REMASTERED,2018-05-23,2000000 - 5000000,2879,17,39.99,0,"Then, there was fire. Re-experience the critic...","[English, French, Italian, German, Spanish - S...","[English, Traditional Chinese]",,True,False,False,84,https://www.metacritic.com/game/pc/dark-souls-...,0,119152,10147,0,41,91117,2473,338,1615,297,"QLOC, FromSoftware, Inc.","FromSoftware, Inc., Bandai Namco Entertainment","Single-player, Multi-player, Steam Achievement...",Action,"Souls-like, Dark Fantasy, RPG, Difficult, Acti...",2018,129299,92.152298,[Action],Action,False,2000000.0,79980000.0,"[Souls-like, Dark Fantasy, RPG, Difficult, Act...",[Action],23,5,2018
9038,335300,DARK SOULS™ II: Scholar of the First Sin,2015-04-01,2000000 - 5000000,1612,13,39.99,0,DARK SOULS™ II: Scholar of the First Sin bring...,"[English, French, Italian, German, Spanish - S...","[English, Traditional Chinese]",“... an even better version of one of our favo...,True,False,False,79,https://www.metacritic.com/game/pc/dark-souls-...,0,98052,18658,0,38,78296,3247,1118,1830,1360,"FromSoftware, Inc.","Bandai Namco Entertainment, FromSoftware, Inc.","Single-player, Multi-player, Co-op, Steam Achi...","Action, RPG","Souls-like, Dark Fantasy, RPG, Difficult, Acti...",2015,116710,84.013366,"[Action, RPG]",Action,False,2000000.0,79980000.0,"[Souls-like, Dark Fantasy, RPG, Difficult, Act...","[Action, RPG]",1,4,2015
35526,236430,DARK SOULS™ II,2014-04-25,500000 - 1000000,197,0,39.99,4,"Developed by FROM SOFTWARE, DARK SOULS™ II is ...","[English, French, Italian, German, Spanish - S...","[English, Traditional Chinese]",“…this is the Dark Souls sequel PC gamers dese...,True,False,False,91,https://www.metacritic.com/game/pc/dark-souls-...,0,39201,5123,0,38,31602,4980,85,2772,85,"FromSoftware, Inc.","BANDAI NAMCO Entertainment, FromSoftware, Inc.","Single-player, Multi-player, Co-op, Steam Achi...","Action, RPG","Souls-like, RPG, Dark Fantasy, Difficult, Acti...",2014,44324,88.441928,"[Action, RPG]",Action,False,500000.0,19995000.0,"[Souls-like, RPG, Dark Fantasy, Difficult, Act...","[Action, RPG]",25,4,2014


In [32]:
import plotly.express as px

# 1. Contar cuántos juegos se lanzaron por año
# Eliminamos filas sin año y filtramos hasta 2026 (para evitar datos erróneos del futuro)
df_juegos_por_año = df[df['Release_Year'] < 2026]['Release_Year'].value_counts().reset_index()

# Renombrar las columnas para que el gráfico quede limpio
df_juegos_por_año.columns = ['Año', 'Cantidad de Juegos']

# Ordenar por año cronológicamente
df_juegos_por_año = df_juegos_por_año.sort_values(by='Año')

# 2. Crear el gráfico de barras con Plotly
fig = px.bar(
    df_juegos_por_año, 
    x="Año", 
    y="Cantidad de Juegos",
    title="Cantidad de Juegos Lanzados por Año en Steam",
    labels={"Año": "Año de Lanzamiento", "Cantidad de Juegos": "Número de Títulos"},
    text_auto=True,    # Muestra el número exacto encima de cada barra
    template="plotly_dark"
)

# Ajustes estéticos para que el eje X muestre bien los años
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))

# Mostrar gráfico
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# 2. Filtrar años recientes y válidos
df_filtrado = df[(df['Release_Year'] >= 2014) & (df['Release_Year'] < 2026)].copy()

df_filtrado['Genres'] = df_filtrado['Genres'].fillna('').astype(str)

df_filtrado['Genre_List'] = df_filtrado['Genres'].apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip() not in ['', 'nan']]
)


# 'Explode' duplica las filas para que cada etiqueta/género tenga su propia fila
df_exploded = df_filtrado.explode('Genre_List')

# Eliminamos filas vacías que hayan podido quedar
df_exploded = df_exploded.dropna(subset=['Genre_List'])
df_exploded = df_exploded[df_exploded['Genre_List'] != '']

# --- OPCIONAL: Filtro de limpieza para omitir términos técnicos si se cuelan ---
# Si ves que aparecen cosas como '2d' o '3d' y solo quieres géneros puros, 
# puedes descomentar las líneas de abajo para saltártelos:
# palabras_a_omitir = ['singleplayer', 'multiplayer', '2d', '3d', 'atmospheric', 'story rich']
# df_exploded = df_exploded[~df_exploded['Genre_List'].str.lower().isin(palabras_a_omitir)]

# 4. Agrupar por Año y Género/Tag para contar cuántos juegos hay de cada uno
df_counts = df_exploded.groupby(['Release_Year', 'Genre_List']).size().reset_index(name='Cantidad')

# 5. Filtrar para quedarnos SOLO con los TOP 5 de cada año
df_top5 = df_counts.sort_values(['Release_Year', 'Cantidad'], ascending=[True, False])
df_top5 = df_top5.groupby('Release_Year').head(5)

# 6. Crear el gráfico de barras agrupado con Plotly
fig = px.bar(
    df_top5, 
    x="Release_Year", 
    y="Cantidad", 
    color="Genre_List",              # Los colores representan los diferentes géneros reales
    barmode="group",                 # Las barras se colocan una al lado de la otra por año
    title="Top 5 Tipos de Juegos Lanzados por Año en Steam (2014 - 2025)",
    labels={"Release_Year": "Año", "Cantidad": "Número de Juegos", "Genre_List": "Género/Etiqueta"},
    template="plotly_dark",
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Ajustes visuales para el eje X
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig.show()

In [ ]:
print(df_exploded)

          Appid                                   Name Release_date  \
1        496350  Supipara - Chapter 1 Spring Has Come!   2016-07-29   
2       1034400      Mystery Solitaire The Black Raven   2019-05-06   
3       3292190            버튜버 파라노이아 - Vtuber Paranoia   2024-10-31   
3       3292190            버튜버 파라노이아 - Vtuber Paranoia   2024-10-31   
3       3292190            버튜버 파라노이아 - Vtuber Paranoia   2024-10-31   
...         ...                                    ...          ...   
122576  4183380                            Fall Asleep   2025-12-31   
122591  4217800                                Sweeper   2025-12-31   
122591  4217800                                Sweeper   2025-12-31   
122593  2071500                     Islands of Insight   2024-02-13   
122593  2071500                     Islands of Insight   2024-02-13   

       Estimated_owners  Peak_ccu  Required_age  Price  Dlc_count  \
1             0 - 20000         0             0   5.24          0   
2        

In [ ]:
import os
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# ==========================================
# 1. PREPARACIÓN DE LOS DATOS
# ==========================================

# Aseguramos que la columna Genres sea texto y esté limpia
df['Genres'] = df['Genres'].fillna('').astype(str)

# Filtramos años válidos para el menú
años_validos = sorted([int(año) for año in df['Release_Year'].dropna().unique() if 2010 <= año <= 2026])

# Hacemos el explode una sola vez para que la app sea rápida
df_exploded = df.copy()
df_exploded['Genre_List'] = df_exploded['Genres'].apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip() not in ['', 'nan']]
)

df_exploded = df_exploded.explode('Genre_List')
df_exploded = df_exploded.dropna(subset=['Genre_List'])
df_exploded = df_exploded[df_exploded['Genre_List'] != '']


# ==========================================
# 2. DISEÑO DE LA INTERFAZ (DASH APP)
# ==========================================
app = dash.Dash(__name__)

app.layout = html.Div(style={'backgroundColor': '#111111', 'color': '#FFFFFF', 'padding': '20px', 'fontFamily': 'Arial'}, children=[
    html.H1("Distribución de Géneros de Steam por Año", style={'textAlign': 'center', 'marginBottom': '30px'}),
    
    html.Div(style={'display': 'flex', 'justifyContent': 'space-around', 'marginBottom': '30px'}, children=[
        # Selector de Año
        html.Div([
            html.Label("Selecciona el Año de Lanzamiento:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='selector-año',
                options=[{'label': str(a), 'value': a} for a in años_validos],
                value=2024,
                style={'color': '#000000', 'width': '250px', 'marginTop': '5px'}
            )
        ]),
        
        # Selector de Cantidad de Géneros (Top X)
        html.Div([
            html.Label("Cantidad de Géneros a mostrar (Top X):", style={'fontWeight': 'bold'}),
            dcc.Slider(
                id='slider-genres',
                min=3,
                max=15,
                step=1,
                value=8,
                marks={
                    i: {
                        'label': str(i),
                        'style': {'color': '#FFFFFF', 'fontSize': '14px'}
                    }
                    for i in range(3, 16, 2)
                },
            )
        ], style={'width': '40%'})
    ]),
    
    # Contenedor del Gráfico
    html.Div([
        dcc.Graph(id='pie-chart-generos')
    ])
])


# ==========================================
# 3. CALLBACK — LÓGICA DEL GRÁFICO
# ==========================================
@app.callback(
    Output('pie-chart-generos', 'figure'),
    [Input('selector-año', 'value'),
     Input('slider-genres', 'value')]
)
def actualizar_grafico(año_seleccionado, top_x):
    # 1. Filtrar por año
    df_año = df_exploded[df_exploded['Release_Year'] == año_seleccionado]

    # Total de juegos únicos ese año
    total_juegos = df_año['Appid'].nunique()
    
    # 2. Contar géneros
    conteo_gen = df_año['Genre_List'].value_counts().reset_index()
    conteo_gen.columns = ['Genero', 'Cantidad']
    
    # 3. Top X
    df_top = conteo_gen.head(top_x).copy()
    
    # 4. Calcular "Otros"
    total_menciones = conteo_gen['Cantidad'].sum()
    suma_top = df_top['Cantidad'].sum()
    cantidad_otros = total_menciones - suma_top
    
    fila_otros = pd.DataFrame([{'Genero': 'Otros Géneros', 'Cantidad': cantidad_otros}])
    df_final = pd.concat([df_top, fila_otros], ignore_index=True)
    
    # 5. Pie Chart
    fig = px.pie(
        df_final,
        values='Cantidad',
        names='Genero',
        title=f"Distribución de Géneros en {año_seleccionado} (Top {top_x})",
        template="plotly_dark",
        hole=0.4
    )
    
    fig.update_traces(
        textinfo='percent+label',
        textposition='inside',
        hovertemplate="<b>%{label}</b><br>Menciones: %{value}<br>Porcentaje: %{percent}<extra></extra>"
    )
    
    fig.update_layout(
        transition_duration=500,
        title={
            'text': f"Géneros en {año_seleccionado} (Top {top_x})"
                    f"<br><sup>Total juegos lanzados: {total_juegos}</sup>",
            'x': 0.5
        }
    )
    
    return fig


# ==========================================
# 4. EJECUTAR LA APP
# ==========================================
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)


In [ ]:
all_g = []
genres =df["Genres"].dropna().values
for g in genres:
    for word in g.split(",") :
        all_g.append(word)


genres_not_unique = pd.DataFrame(all_g , columns=["genres"])
genres_unique_counts = genres_not_unique.groupby(['genres'])['genres'].count()
genres_unique_counts = genres_unique_counts.sort_values(ascending=False)

In [ ]:
df["Genres"].value_counts()

Genres
                                                                                                 8413
Casual, Indie                                                                                    6700
Action, Indie                                                                                    5679
Action, Adventure, Indie                                                                         5159
Adventure, Indie                                                                                 4311
                                                                                                 ... 
Gore, Indie, RPG                                                                                    1
Action, Adventure, Casual, Massively Multiplayer, RPG, Simulation, Free To Play, Early Access       1
Action, Casual, RPG, Simulation, Sports, Strategy, Early Access                                     1
Action, Adventure, Casual, Racing, RPG, Sports                             

In [ ]:
genres_unique_counts.head(10)

genres
 Indie           69907
Action           45888
 Casual          26384
Adventure        24053
Casual           23826
 Simulation      21665
 Adventure       21088
 Strategy        20815
 RPG             19062
 Free To Play    11302
Name: genres, dtype: int64

In [ ]:
print(df.head())

     Appid                                   Name Release_date  \
0  2539430             Black Dragon Mage Playtest   2023-08-01   
1   496350  Supipara - Chapter 1 Spring Has Come!   2016-07-29   
2  1034400      Mystery Solitaire The Black Raven   2019-05-06   
3  3292190            버튜버 파라노이아 - Vtuber Paranoia   2024-10-31   
4  3631080                          Maze Quest VR   2025-04-24   

  Estimated_owners  Peak_ccu  Required_age  Price  Dlc_count  \
0            0 - 0         0             0   0.00          0   
1        0 - 20000         0             0   5.24          0   
2        0 - 20000         0             0   4.99          0   
3        0 - 20000         1             0   8.99          1   
4        0 - 20000         0             0   4.99          0   

                                   Short_description  \
0                                                NaN   
1  Spring has come, and our protagonist, Yukinari...   
2      Discover an entrancing and spectacular worl

In [ ]:
df.Windows.value_counts()

Windows
True     122567
False        44
Name: count, dtype: int64

In [ ]:
# Asegurar que Release_Year es entero
df['Release_Year'] = pd.to_numeric(df['Release_Year'], errors='coerce')

# Filtrar años válidos
df_years = df[df['Release_Year'].between(2010, 2026)]

# Crear columna "All_3" para juegos que soportan Windows, Mac y Linux
df_years['All_3'] = (
    df_years['Windows'].astype(bool) &
    df_years['Mac'].astype(bool) &
    df_years['Linux'].astype(bool)
)
df_years['Windows'] = df_years['Windows'].astype(str).str.lower().isin(['true', '1'])
df_years['Mac'] = df_years['Mac'].astype(str).str.lower().isin(['true', '1'])
df_years['Linux'] = df_years['Linux'].astype(str).str.lower().isin(['true', '1'])

# Agrupar por año
df_platforms = df_years.groupby('Release_Year').agg({
    'Windows': 'sum',
    'Mac': 'sum',
    'Linux': 'sum',
    'All_3': 'sum'
}).reset_index()

fig = px.line(
    df_platforms,
    x='Release_Year',
    y=['Windows', 'Mac', 'Linux', 'All_3'],
    markers=True,
    title="Número de juegos por año según soporte de plataforma",
    template="plotly_dark"
)

fig.update_layout(
    xaxis_title="Año",
    yaxis_title="Número de juegos",
    legend_title="Plataforma"
)

fig.show()

